<a href="https://colab.research.google.com/github/chetools/CHE4071_Spring2026/blob/main/pH_control.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!wget -N -q https://raw.githubusercontent.com/chetools/chetools/main/tools/che5.ipynb -O che5.ipynb
%run che5.ipynb

In [132]:
def netH(pH):
    H = 10**-pH
    OH = 1e-14 / H
    netH = H - OH
    return netH
    # return jnp.where(pH<7, 10**(-pH), -10**(-14+pH))

def pH(netH):
    #x = 1e-14/(netH + x), x=H+ and OH- from H2O hydrolysis and charge balance
    #x**2 + netH*x -1e-14 = 0
    d= jnp.sqrt(netH**2 + 4*1e-14)
    x = jnp.where(netH>0, (-netH + d)/2, (-netH - d)/2)
    return jnp.squeeze(jnp.where(netH>0, -jnp.log10(netH+x), 14.+jnp.log10(-netH-x)))

dpH_netH = jax.grad(pH)

In [133]:
def make_step(t0):
    def step(t):
        return jnp.where(t>t0, 1, 0.)
    return step

In [193]:
V = 1.  #L
pHA, pHB = 1., 13.
Kc = 1e6

In [194]:
def dpHdt(t, pH, u):
    qin, pHin = u
    netH_err = netH(pH_setpoint)-netH(pH)
    qA = jnp.where(netH_err>0, Kc*jnp.abs(netH_err), 0.)
    qB = jnp.where(netH_err<0, Kc*jnp.abs(netH_err), 0.)

    dnetH_dt = (qin*netH(pHin) + qA*netH(pHA) + qB*netH(pHB) - (qin+qA+qB)*netH(pH))/V
    return dpH_netH(pH) * dnetH_dt

In [195]:
step1 = make_step(1000.)
step2 = make_step(2000.)
step3 = make_step(1500)
step4 = make_step(2500)

#disturbance in inlet pH
def dpHdt2(t, pH, u):
    qin = u
    pHin = 6.
    pH_setpoint = 6 + 2*(step3(t) - step4(t))
    netH_err = netH(pH_setpoint)-netH(pH)
    qA = jnp.where(netH_err>0, Kc*jnp.abs(netH_err), 0.)
    qB = jnp.where(netH_err<0, Kc*jnp.abs(netH_err), 0.)
    dnetH_dt = (qin*netH(pHin) + qA*netH(pHA) + qB*netH(pHB) - (qin+qA+qB)*netH(pH))/V
    return dpH_netH(pH) * dnetH_dt

In [196]:
u = jnp.array([5.])

In [199]:
tend = 4000.
tplot = np.linspace(0,tend,100)
res=sp.integrate.solve_ivp(lambda t, pH: dpHdt2(t, pH, u), (0, tend), jnp.array([7.]), method='Radau', dense_output=True)
pHplot = res.sol(tplot)[0]

In [200]:
fig=make_subplots()
fig.add_scatter(x=tplot, y=pHplot)
fig.update_layout(width=600, height=400)